## Customer retention: first baseline

This notebook creates historical customer snapshots and evaluates a simple ranking on a later period. A customer is eligible after buying in the previous 180 days; the outcome is whether they buy again in the following 90 days. No intervention effect is estimated.


## 1. Load orders

Read both years of the UCI workbook, keep identifiable positive purchases, and combine product lines into invoices. This repeats the documented cleaning rules from the data inspection notebook so this notebook can run on its own.


In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path("data/raw/online_retail_II.xlsx")
if not file_path.exists():
    file_path = Path("../data/raw/online_retail_II.xlsx")
if not file_path.exists():
    raise FileNotFoundError("Download Online Retail II and place online_retail_II.xlsx in data/raw/")

workbook = pd.ExcelFile(file_path)
transactions = pd.concat(
    [pd.read_excel(workbook, sheet_name=name) for name in workbook.sheet_names],
    ignore_index=True,
)
transactions["InvoiceDate"] = pd.to_datetime(transactions["InvoiceDate"], errors="coerce")

valid_purchases = transactions.loc[
    transactions["Customer ID"].notna()
    & transactions["InvoiceDate"].notna()
    & ~transactions["Invoice"].astype(str).str.upper().str.startswith("C")
    & (transactions["Quantity"] > 0)
    & (transactions["Price"] > 0)
].copy()
valid_purchases["line_revenue"] = valid_purchases["Quantity"] * valid_purchases["Price"]

orders = (
    valid_purchases.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(order_date=("InvoiceDate", "min"), order_value=("line_revenue", "sum"))
)
print("Valid purchase rows:", len(valid_purchases))
print("Orders:", len(orders))
print("Customers:", orders["Customer ID"].nunique())

## 2. Build one customer snapshot

At each cutoff, compute recency, order count, and spending using only earlier orders. The 90-day outcome is calculated from later orders and stored in a separate label column.


In [ ]:
def build_cohort(orders, cutoff):
    cutoff = pd.Timestamp(cutoff)
    history_start = cutoff - pd.Timedelta(days=180)
    outcome_end = cutoff + pd.Timedelta(days=90)

    history = orders.loc[
        orders["order_date"].ge(history_start)
        & orders["order_date"].lt(cutoff)
    ]
    future = orders.loc[
        orders["order_date"].ge(cutoff)
        & orders["order_date"].lt(outcome_end)
    ]

    cohort = history.groupby("Customer ID").agg(
        last_order_date=("order_date", "max"),
        order_count_180d=("Invoice", "nunique"),
        spend_180d=("order_value", "sum"),
        average_order_value=("order_value", "mean"),
    )
    cohort["days_since_last_order"] = (cutoff - cohort["last_order_date"]).dt.days
    cohort["returned_90d"] = cohort.index.isin(future["Customer ID"]).astype(int)
    cohort["did_not_return_90d"] = 1 - cohort["returned_90d"]
    return cohort.drop(columns="last_order_date").reset_index().assign(cutoff=cutoff)

example = build_cohort(orders, "2011-06-01")
print("June 2011 customers:", len(example))
print("Returned within 90 days:", example["returned_90d"].sum())
display(example.head())

## 3. Separate earlier and later periods

Create six snapshots. The first five provide development data; September 2011 is held out as the later test period. Customers can appear in several snapshots, so this evaluates a future period rather than performance on entirely new customers.


In [ ]:
cutoffs = pd.to_datetime([
    "2010-06-01", "2010-09-01", "2010-12-01",
    "2011-03-01", "2011-06-01", "2011-09-01",
])
assert cutoffs[-1] + pd.Timedelta(days=90) <= orders["order_date"].max()

snapshots = pd.concat([build_cohort(orders, date) for date in cutoffs], ignore_index=True)
train = snapshots.loc[snapshots["cutoff"] < pd.Timestamp("2011-09-01")].copy()
test = snapshots.loc[snapshots["cutoff"] == pd.Timestamp("2011-09-01")].copy()

summary = snapshots.groupby("cutoff").agg(
    customers=("Customer ID", "size"),
    return_share=("returned_90d", "mean"),
)
print("Development rows:", len(train), "| Later test rows:", len(test))
display(summary)

## 4. Evaluate a recency baseline

Rank customers by days since their last purchase, with longer gaps treated as greater nonreturn risk. Measure how well this simple rule ranks the later test period. The top-20% figures describe prioritization, not the effect of contacting anyone.


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

actual = test["did_not_return_90d"]
risk_score = test["days_since_last_order"]
selected_count = int(np.ceil(0.20 * len(test)))
priority = test.nlargest(selected_count, "days_since_last_order")

results = pd.Series({
    "nonreturn_share": actual.mean(),
    "roc_auc": roc_auc_score(actual, risk_score),
    "average_precision": average_precision_score(actual, risk_score),
    "top_20pct_nonreturn_share": priority["did_not_return_90d"].mean(),
    "top_20pct_nonreturn_capture": priority["did_not_return_90d"].sum() / actual.sum(),
})
print("Later test period: September 1 to November 30, 2011")
print("Customers prioritized:", selected_count, "of", len(test))
display(results.round(3))